# Logistic Regression (gap direction)\n\nGoal: predict whether the weekend gap is **up** (`y_gap_up=1`) or **down/flat** (`0`).\n\nSamples are built as **Friday features → next Monday target**, to avoid leakage.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent.parent  # notebooks/models -> repo root
WEEKEND_CSV = REPO_ROOT / "structured_csv_data_files" / "Weekend_dataset.csv"
WEEKEND_CSV

In [ ]:
w = pd.read_csv(WEEKEND_CSV)
w["Date_parsed"] = pd.to_datetime(w["Date"], errors="coerce", utc=True)
w["weekday_num"] = w["Date_parsed"].dt.weekday

for c in ["Open", "Close", "High", "Low", "Volume"]:
    if c in w.columns:
        w[c] = pd.to_numeric(w[c], errors="coerce")

w = w.dropna(subset=["Ticker", "Date_parsed"]).sort_values(["Ticker", "Date_parsed"]).reset_index(drop=True)

# next-row join per ticker
w["next_weekday"] = w.groupby("Ticker")["weekday_num"].shift(-1)
w["next_date"] = w.groupby("Ticker")["Date_parsed"].shift(-1)
w["next_open"] = w.groupby("Ticker")["Open"].shift(-1)

pairs = w.loc[(w["weekday_num"] == 4) & (w["next_weekday"] == 0)].copy()
delta_days = (pairs["next_date"] - pairs["Date_parsed"]).dt.days
pairs = pairs.loc[(delta_days >= 3) & (delta_days <= 5)].copy()

pairs["y_gap_pct"] = (pairs["next_open"] - pairs["Close"]) / pairs["Close"]
pairs["y_gap_up"] = (pairs["y_gap_pct"] > 0).astype(int)

pairs[["Ticker", "Date_parsed", "next_date", "Close", "next_open", "y_gap_pct", "y_gap_up"]].head()

In [ ]:
# Define feature columns (Friday-only)
exclude = {
    "Date",
    "Date_parsed",
    "weekday_num",
    "next_weekday",
    "next_date",
    "next_open",
    "y_gap_pct",
    "y_gap_up",
}

feature_cols = [c for c in pairs.columns if c not in exclude]

# Keep only numeric features for Logistic Regression
X_all = pairs[feature_cols].select_dtypes(include=["number"]).copy()
y_all = pairs["y_gap_up"].copy()

X_all.shape, y_all.value_counts()

## Train/test split (time-based)\n\nWe split by **Monday date** to simulate forecasting forward in time.

In [ ]:
cutoff = pairs["next_date"].quantile(0.8)  # 80% oldest -> train, newest -> test

train_idx = pairs["next_date"] <= cutoff
test_idx = ~train_idx

X_train, y_train = X_all.loc[train_idx], y_all.loc[train_idx]
X_test, y_test = X_all.loc[test_idx], y_all.loc[test_idx]

X_train.shape, X_test.shape, cutoff

In [ ]:
# Model: Logistic Regression (with scaling + imputation)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("accuracy:", accuracy_score(y_test, pred))
print("roc_auc:", roc_auc_score(y_test, proba))
print("confusion_matrix:\n", confusion_matrix(y_test, pred))
print("\nclassification_report:\n", classification_report(y_test, pred))

In [ ]:
# Coefficients (rough feature importance) 
feature_names = X_train.columns.to_list()
coefs = clf.named_steps["logreg"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs}).sort_values("coef", key=lambda s: s.abs(), ascending=False)
coef_df.head(30)